In [3]:
import os
import pickle
import networkx as nx
import sys
import pandas as pd

# Determine the project root directory for relative imports
try:
    # This will work in scripts where __file__ is defined
    current_dir = os.path.dirname(os.path.abspath(__file__))
    # Assuming "src" is parallel to the script folder
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
except NameError:
    # In notebooks __file__ is not defined: assume we're in notebooks/riziv_dataset/
    project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Local application imports


In [4]:
# Define the path to the BSARD dataset files
BSARD_data_path = os.path.join(project_root, "data", "BSARD_dataset")

bsard_corpus = pd.read_csv(os.path.join(BSARD_data_path, 'inputs', 'bsard_corpus.csv'))

bsard_corpus.head(3)

,id,reference,article,law_type,code,book,part,act,chapter,section,subsection,description
0,1,"Art. 1.1.1, Code Bruxellois de l'Air, du Clima...",Le présent Code règle une matière visée à l'ar...,regional,"Code Bruxellois de l'Air, du Climat et de la M...",Dispositions communes,NaN,Généralités,NaN,NaN,NaN,"Dispositions communes, Généralités"
1,2,"Art. 1.1.2, Code Bruxellois de l'Air, du Clima...",Le présent Code transpose en Région de Bruxell...,regional,"Code Bruxellois de l'Air, du Climat et de la M...",Dispositions communes,NaN,Généralités,NaN,NaN,NaN,"Dispositions communes, Généralités"
2,3,"Art. 1.2.1, Code Bruxellois de l'Air, du Clima...",Le présent Code poursuit les objectifs suivant...,regional,"Code Bruxellois de l'Air, du Climat et de la M...",Dispositions communes,NaN,Objectifs,NaN,NaN,NaN,"Dispositions communes, Objectifs"


In [ ]:
base_document_graph_path = os.path.join(BSARD_data_path, 'intermediate', 'base_document_graph_V2.pkl')

# Carga del grafo
with open(base_document_graph_path, 'rb') as f:
    G = pickle.load(f)

In [6]:
non_document_entities_path = os.path.join(BSARD_data_path, 'intermediate', 'keyword_extraction.pkl')

# Carga del diccionario de keywords
with open(non_document_entities_path, 'rb') as f:
    keywords_dict = pickle.load(f)

In [8]:
###############
# Parse retrieved keyword-related content and build dataframe
###############

# Keep only those articles which have been already scanned
keywords_dict_filter = {k:v for k,v in keywords_dict.items() if v != None} 

parsed_key_terms = []

for art in keywords_dict_filter.keys():

    for n in ["1", "2", "3", "4"]:

        parsed_key_terms.append((art, keywords_dict_filter[art][f"key_concept_{n}"]))

# Build Dataframe
key_terms_df = pd.DataFrame(parsed_key_terms, columns=['article_code', 'key_term'])

In [11]:
filtered_counts = key_terms_df["key_term"].value_counts()[key_terms_df["key_term"].value_counts() > 1]

In [12]:
filtered_counts

key_term
AWIPH                              261
Gouvernement                       140
Code décrétal                      135
assemblée générale                 128
procès-verbal                      121
                                  ... 
état détaillé de la liquidation      2
jury de dégustation                  2
biens contrefaisants                 2
fin anticipée                        2
mode de liquidation                  2
Name: count, Length: 11890, dtype: int64

In [16]:
filtered_keyterms_df = key_terms_df[key_terms_df['key_term'].isin(filtered_counts.index)]
filtered_keyterms_df = filtered_keyterms_df.reset_index(drop=True)

In [24]:
filtered_keyterms_df

,article_code,key_term
0,1.1.1.1,Constitution
1,1.1.1.1,Code
2,1.1.1.1,présent Code
3,1.1.1.2,Directive 2003/87/CE
4,1.1.1.2,Qualité de l'air ambiant
...,...,...
53622,35.0.9.8,Gouvernement fédéral
53623,35.0.10.2,révision de la Constitution
53624,35.0.10.2,temps de guerre
53625,35.0.10.3,pouvoirs constitutionnels du Roi


In [31]:
df_keyterms_condensed = (
    key_terms_df
    .groupby('key_term')
    .agg(
        article_codes=('article_code', list),
        count=('article_code', 'size'),
        law_count=('article_code',lambda s: s.str.split('.').str[0].nunique())
    )
    .reset_index()
)

df_keyterms_condensed = df_keyterms_condensed[df_keyterms_condensed['count'] > 1]
df_keyterms_condensed.reset_index(drop=True, inplace=True)
df_keyterms_condensed

,key_term,article_codes,count,law_count
0,10 septembre,"[28.1.3.36, 28.1.3.37]",2,1
1,17 mars 2001,"[27.0.0.8, 27.0.0.10]",2,1
2,1er janvier,"[16.2.4.6, 16.7.3.2, 20.1.3.3, 27.0.6.73, 27.2...",5,3
3,1er janvier 2003,"[16.6.17.24, 29.4.2.45]",2,2
4,1er janvier 2011,"[16.6.7.12, 16.6.17.1]",2,1
...,...,...,...,...
11885,événement futur et incertain,"[4.0.2.67, 4.0.2.80]",2,1
11886,événement inertain,"[4.4.3.126, 4.0.12.1]",2,1
11887,événements de force majeure,"[25.7.8.73, 25.7.8.159]",2,1
11888,événements exceptionnels,"[21.0.2.45, 21.0.2.46, 33.4.2.14]",3,2


In [33]:
top10 = df_keyterms_condensed.nlargest(10, 'law_count')[['key_term','law_count']]
top10

,key_term,law_count
9503,recours,21
8859,procès-verbal,20
9407,rapport annuel,19
9178,présent chapitre,18
354,Code judiciaire,17
2158,amende administrative,17
7786,notification de la décision,17
2653,autorisation préalable,16
7777,notification,16
8349,personne morale,16


In [34]:
# Assuming you have:
# - summary_df: a DataFrame with columns 'key_term' and 'article_codes'
# - G: your pre-built hierarchical graph of Laws → Books → Chapters → Articles

# 1) Add each key term as a new node with node_type="KeyTerm"
G.add_nodes_from(
    (key_term, {"node_type": "KeyTerm"})
    for key_term in df_keyterms_condensed['key_term']
)

# 2) Create edges between articles and key terms:
#    article -> key_term  with relation="cites"
#    key_term -> article  with relation="cited_in"
G.add_edges_from(
    (article_code, key_term, {"relation": "cites"})
    for key_term, codes in zip(df_keyterms_condensed['key_term'], df_keyterms_condensed['article_codes'])
    for article_code in codes
)
G.add_edges_from(
    (key_term, article_code, {"relation": "cited_in"})
    for key_term, codes in zip(df_keyterms_condensed['key_term'], df_keyterms_condensed['article_codes'])
    for article_code in codes
)


In [37]:
G.nodes["10 septembre"]

{'node_type': 'KeyTerm'}

In [40]:
list(G.neighbors("10 septembre"))

['28.1.3.36', '28.1.3.37']

In [41]:
list(G.neighbors('28.1.3.37'))

['28.1.3',
 '28.1.3.36',
 '28.1.3.38',
 '10 septembre',
 'centre de vote adapté',
 'déclaration auprès de la commune',
 'mobilité réduite']

In [42]:
len(G.nodes())

35496

In [43]:
from collections import Counter
import networkx as nx

# Basic counts
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# Count nodes by type
node_types = [data.get('node_type', 'Unknown') for _, data in G.nodes(data=True)]
nodes_by_type = Counter(node_types)

# Degree metrics
degrees = dict(G.degree())
avg_degree = sum(degrees.values()) / num_nodes if num_nodes else 0
max_degree = max(degrees.values()) if degrees else 0
min_degree = min(degrees.values()) if degrees else 0

# (Optional) If directed, you can also look at in/out-degree:
if G.is_directed():
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    avg_in_degree = sum(in_deg.values()) / num_nodes
    avg_out_degree = sum(out_deg.values()) / num_nodes

# Assemble and print summary
summary = {
    "total_nodes": num_nodes,
    "total_edges": num_edges,
    "nodes_by_type": dict(nodes_by_type),
    "avg_degree": avg_degree,
    "max_degree": max_degree,
    "min_degree": min_degree,
}

if G.is_directed():
    summary.update({
        "avg_in_degree": avg_in_degree,
        "avg_out_degree": avg_out_degree
    })

print("Graph summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")


Graph summary:
  total_nodes: 35496
  total_edges: 198112
  nodes_by_type: {'Central Node': 1, 'Act': 35, 'Book': 150, 'Title': 795, 'Article': 22625, 'KeyTerm': 11890}
  avg_degree: 11.16249718278116
  max_degree: 1080
  min_degree: 2
  avg_in_degree: 5.58124859139058
  avg_out_degree: 5.58124859139058


## 5. Save output

In [46]:
with open(os.path.join(BSARD_data_path, 'intermediate', "hybrid_graph_full.pkl"), 'wb') as f:
    pickle.dump(G, f)